# Task 1 v2: Learn `u` and `gradU` (Instantaneous Surrogate)

This notebook implements the mode you requested:
- input at time `t`: particle state `x_t`
- output at time `t`: `[u_t, gradU_t]`
- FLOWUnsteady still does O(N) state integration and time marching
- the ML model replaces only expensive velocity/gradient evaluation


In [ ]:

from pathlib import Path
import json
import time
import random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

NOTEBOOK_VERSION = 'task1_ugradu_v2_weighted_2026-05-10'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Notebook version:', NOTEBOOK_VERSION)

CWD = Path.cwd().resolve()
if (CWD / 'final-2' / 'output').exists():
    BASE = CWD / 'final-2'
elif (CWD.name == 'notebooks') and (CWD.parent / 'output').exists():
    BASE = CWD.parent
else:
    BASE = CWD

DATA_PATH = BASE / 'output' / 'particle_ugradu_dataset.npz'
OUT_DIR = BASE / 'output' / 'task1_ugradu_training'
OUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Base:', BASE)
print('DATA_PATH:', DATA_PATH)
print('Exists?:', DATA_PATH.exists())
print('Device:', DEVICE)
if not DATA_PATH.exists():
    raise FileNotFoundError('Run preprocess_data.py with TASK1_TARGET_MODE="ugradu" first')


In [ ]:

# Load dataset

ds = np.load(DATA_PATH, allow_pickle=True)
X_all_norm = ds['inputs_t_norm'].astype(np.float32)
Y_norm = ds['targets_ugradu_norm'].astype(np.float32)
X_all_raw = ds['inputs_t'].astype(np.float32)
Y_raw = ds['targets_ugradu'].astype(np.float32)

feature_names_all = [str(x) for x in ds['feature_names'].tolist()]
target_names = [str(x) for x in ds['target_names'].tolist()]

frame_ranges = list(ds['frame_ranges'])
frame_contexts = list(ds['frame_contexts'])
train_frame_ids = ds['train_frame_ids'].astype(np.int64)
val_frame_ids = ds['val_frame_ids'].astype(np.int64)
test_frame_ids = ds['test_frame_ids'].astype(np.int64)

train_rows = ds['train_rows'].astype(np.int64)
val_rows = ds['val_rows'].astype(np.int64)
test_rows = ds['test_rows'].astype(np.int64)

train_cases = [str(x) for x in ds['train_cases'].tolist()]
val_cases = [str(x) for x in ds['val_cases'].tolist()]
test_cases = [str(x) for x in ds['test_cases'].tolist()]

out_mean = ds['out_mean'].astype(np.float32)
out_std = ds['out_std'].astype(np.float32)

# Geometry channel ablation switch
# 'all'       -> use all input channels
# 'dist_only' -> keep only geom_dist from geometry channels
# 'none'      -> remove all geometry channels
GEOM_MODE = 'all'

geom_ch = [k for k in feature_names_all if k.startswith('geom_')]
if GEOM_MODE == 'all':
    keep_features = feature_names_all
elif GEOM_MODE == 'dist_only':
    keep_features = [k for k in feature_names_all if (not k.startswith('geom_')) or (k == 'geom_dist')]
elif GEOM_MODE == 'none':
    keep_features = [k for k in feature_names_all if not k.startswith('geom_')]
else:
    raise ValueError(f'Unknown GEOM_MODE={GEOM_MODE}')

keep_idx = np.array([feature_names_all.index(k) for k in keep_features], dtype=np.int64)
X_norm = X_all_norm[:, keep_idx]
X_raw = X_all_raw[:, keep_idx]
feature_names = keep_features

geom_dist_idx = feature_names.index('geom_dist') if 'geom_dist' in feature_names else None

print('inputs_t_norm shape (all) :', X_all_norm.shape)
print('inputs_t_norm shape (used):', X_norm.shape)
print('targets_ugradu_norm shape :', Y_norm.shape)
print('n_frames                  :', len(frame_ranges))
print('feature_names (used)      :', feature_names)
print('target_names              :', target_names)
print('geom mode                 :', GEOM_MODE)
print('train/val/test frames     :', len(train_frame_ids), len(val_frame_ids), len(test_frame_ids))
print('train/val/test rows       :', len(train_rows), len(val_rows), len(test_rows))
print('train/val/test cases      :', train_cases, val_cases, test_cases)


In [ ]:
# Dataset class: one sample = one frame (variable particle count)

class FrameDataset(Dataset):
    def __init__(self, X_norm, Y_norm, X_raw, frame_ranges, frame_ids):
        self.X_norm = X_norm
        self.Y_norm = Y_norm
        self.X_raw = X_raw
        self.frame_ranges = frame_ranges
        self.frame_ids = [int(i) for i in frame_ids]

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, idx):
        fid = self.frame_ids[idx]
        case, fr, s, e, n = self.frame_ranges[fid]
        s, e = int(s), int(e)
        x = torch.from_numpy(self.X_norm[s:e])
        y = torch.from_numpy(self.Y_norm[s:e])
        x_raw = torch.from_numpy(self.X_raw[s:e])
        meta = {'frame_id': fid, 'case': str(case), 'fr': str(fr), 'n': int(n)}
        return x, y, x_raw, meta


def collate_frame(batch):
    xs, ys, xrs, ms = zip(*batch)
    return list(xs), list(ys), list(xrs), list(ms)


train_ds = FrameDataset(X_norm, Y_norm, X_raw, frame_ranges, train_frame_ids)
val_ds = FrameDataset(X_norm, Y_norm, X_raw, frame_ranges, val_frame_ids)
test_ds = FrameDataset(X_norm, Y_norm, X_raw, frame_ranges, test_frame_ids)

pin_mem = (DEVICE.type == 'cuda')
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, collate_fn=collate_frame, pin_memory=pin_mem)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, collate_fn=collate_frame, pin_memory=pin_mem)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, collate_fn=collate_frame, pin_memory=pin_mem)

print('dataset lens:', len(train_ds), len(val_ds), len(test_ds))
print('sample frame particle count:', train_ds[0][0].shape[0])



In [ ]:
# Model: GNO for instantaneous u/gradU regression
import inspect

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as exc:
    raise RuntimeError('Need neuralop with GNOBlock') from exc


def rel_l2(pred, tgt, eps=1e-12):
    d = (pred - tgt).reshape(pred.shape[0], -1)
    t = tgt.reshape(tgt.shape[0], -1)
    return (torch.linalg.norm(d, dim=1) / torch.linalg.norm(t, dim=1).clamp_min(eps)).mean()


class UGradUGNO(nn.Module):
    def __init__(self, in_dim, out_dim, hidden=96, n_layers=3, radius=0.12):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden),
        )

        sig = inspect.signature(GNOBlock.__init__)
        p = sig.parameters
        extra = {}
        if 'use_torch_scatter_reduce' in p:
            try:
                import torch_scatter  # noqa: F401
                extra['use_torch_scatter_reduce'] = True
            except Exception:
                extra['use_torch_scatter_reduce'] = False
        if 'use_open3d_neighbor_search' in p:
            try:
                import open3d  # noqa: F401
                extra['use_open3d_neighbor_search'] = True
            except Exception:
                extra['use_open3d_neighbor_search'] = False
        if extra:
            print('GNO backend flags:', extra)

        self.blocks = nn.ModuleList([
            GNOBlock(
                in_channels=hidden,
                out_channels=hidden,
                coord_dim=3,
                radius=radius,
                transform_type='linear',
                reduction='mean',
                pos_embedding_type='transformer',
                pos_embedding_channels=16,
                channel_mlp_layers=[hidden, hidden, hidden],
                **extra,
            ) for _ in range(n_layers)
        ])
        self.norms = nn.ModuleList([nn.LayerNorm(hidden) for _ in range(n_layers)])
        self.head = nn.Sequential(
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, hidden), nn.GELU(),
            nn.Linear(hidden, out_dim),
        )

    def forward(self, x):
        pos = x[:, :3]  # x,y,z
        h = self.enc(x)

        # Reuse neighbor search across blocks with same radius.
        neighbors_by_radius = {}
        for blk, norm in zip(self.blocks, self.norms):
            r = float(blk.radius)
            if r not in neighbors_by_radius:
                neighbors_by_radius[r] = blk.neighbor_search(data=pos, queries=pos, radius=r)
            neighbors = neighbors_by_radius[r]

            pos_in = blk.pos_embedding(pos) if blk.pos_embedding is not None else pos
            u = blk.integral_transform(y=pos_in, x=pos_in, neighbors=neighbors, f_y=h)
            if u.ndim == 3 and u.shape[0] == 1:
                u = u.squeeze(0)
            h = norm(h + u)  # residual block
        return self.head(h)


model = UGradUGNO(X_norm.shape[1], Y_norm.shape[1], hidden=96, n_layers=3, radius=0.12).to(DEVICE)
print('params:', sum(p.numel() for p in model.parameters() if p.requires_grad))



In [ ]:
# Optimizer + training settings
opt_cls = None
try:
    import neuralop.training as nt
    opt_cls = getattr(nt, 'AdamW', None) or getattr(nt, 'Adam', None)
except Exception:
    pass
if opt_cls is None:
    opt_cls = torch.optim.AdamW

EPOCHS = 60
MAX_NODES = 1024
FRAMES_PER_EPOCH = 64
VAL_FRAMES_LIMIT = 24
EVAL_EVERY = 5
TEST_EVERY = 5
PRINT_EVERY_STEPS = 8
GRAD_CLIP_NORM = 1.0

opt = opt_cls(model.parameters(), lr=4e-4, weight_decay=1e-5)
sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-5)

# Loss settings for stiff multi-target regression
LOSS_KIND = 'huber'      # 'huber' | 'mse' | 'logcosh'
HUBER_BETA = 0.08
VEL_WEIGHT = 1.0
GRAD_WEIGHT_STAGE1 = 0.25
GRAD_WEIGHT_STAGE2 = 0.75
GRAD_WEIGHT_STAGE2_EPOCH = 30

# Near-body reweighting: w = 1 + alpha * exp(-dist / scale)
USE_NEAR_BODY_WEIGHT = True
NEAR_ALPHA = 1.5
NEAR_SCALE = 0.20

# Near-body oversampling while downsampling large frames
USE_NEAR_BODY_OVERSAMPLE = False

# Mixed precision for speed on GPU
USE_AMP = (DEVICE.type == 'cuda')
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16
scaler = torch.cuda.amp.GradScaler(enabled=(USE_AMP and AMP_DTYPE == torch.float16))
autocast_kwargs = dict(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP)

print('config:', {
    'EPOCHS': EPOCHS,
    'MAX_NODES': MAX_NODES,
    'FRAMES_PER_EPOCH': FRAMES_PER_EPOCH,
    'VAL_FRAMES_LIMIT': VAL_FRAMES_LIMIT,
    'EVAL_EVERY': EVAL_EVERY,
    'TEST_EVERY': TEST_EVERY,
    'LOSS_KIND': LOSS_KIND,
    'VEL_WEIGHT': VEL_WEIGHT,
    'GRAD_WEIGHT_STAGE1': GRAD_WEIGHT_STAGE1,
    'GRAD_WEIGHT_STAGE2': GRAD_WEIGHT_STAGE2,
    'GRAD_WEIGHT_STAGE2_EPOCH': GRAD_WEIGHT_STAGE2_EPOCH,
    'USE_NEAR_BODY_WEIGHT': USE_NEAR_BODY_WEIGHT,
    'USE_NEAR_BODY_OVERSAMPLE': USE_NEAR_BODY_OVERSAMPLE,
    'USE_AMP': USE_AMP,
    'AMP_DTYPE': str(AMP_DTYPE),
    'GEOM_MODE': GEOM_MODE,
})



In [ ]:
# Training + validation
history = []
best_val = np.inf
best_state = None


def _pointwise_loss(pred, tgt):
    if LOSS_KIND == 'mse':
        return (pred - tgt) ** 2
    if LOSS_KIND == 'logcosh':
        return torch.log(torch.cosh(pred - tgt + 1e-12))
    # default huber
    return F.smooth_l1_loss(pred, tgt, reduction='none', beta=HUBER_BETA)


def _sample_weights_from_raw(x_raw):
    # x_raw is unnormalized selected-feature tensor [N, C]
    if (geom_dist_idx is None) or (not USE_NEAR_BODY_WEIGHT):
        return torch.ones(x_raw.shape[0], device=x_raw.device)
    d = x_raw[:, geom_dist_idx].clamp_min(0.0)
    w = 1.0 + NEAR_ALPHA * torch.exp(-d / NEAR_SCALE)
    return w


def _weighted_channel_mean(loss_nc, w_n):
    # loss_nc: [N, C], w_n: [N]
    wn = w_n / (w_n.mean().clamp_min(1e-12))
    return (loss_nc * wn[:, None]).mean(dim=0)


def split_loss(pred, tgt, x_raw, grad_weight):
    pt = _pointwise_loss(pred, tgt)  # [N,12]
    w = _sample_weights_from_raw(x_raw)
    ch = _weighted_channel_mean(pt, w)

    vel_loss = ch[:3].mean()
    grad_loss = ch[3:].mean()
    total = VEL_WEIGHT * vel_loss + grad_weight * grad_loss

    return total, {
        'total': float(total.item()),
        'vel': float(vel_loss.item()),
        'grad': float(grad_loss.item()),
        'w_mean': float(w.mean().item()),
        'w_max': float(w.max().item()),
    }


def pick_nodes(x, y, x_raw):
    n = x.shape[0]
    if n <= MAX_NODES:
        return x, y, x_raw

    if (geom_dist_idx is None) or (not USE_NEAR_BODY_OVERSAMPLE):
        idx = torch.randperm(n, device=x.device)[:MAX_NODES]
        return x[idx], y[idx], x_raw[idx]

    d = x_raw[:, geom_dist_idx].clamp_min(0.0)
    pr = torch.exp(-d / NEAR_SCALE)
    pr = pr / pr.sum().clamp_min(1e-12)
    idx = torch.multinomial(pr, num_samples=MAX_NODES, replacement=False)
    return x[idx], y[idx], x_raw[idx]


def eval_loader(loader, max_frames=64, grad_weight=1.0):
    model.eval()
    rels, mses = [], []
    vel_rels, grad_rels = [], []
    total_losses, vel_losses, grad_losses = [], [], []

    with torch.no_grad():
        for b, (xs, ys, xrs, ms) in enumerate(loader):
            if b >= max_frames:
                break
            x, y, xr = xs[0].to(DEVICE), ys[0].to(DEVICE), xrs[0].to(DEVICE)
            x, y, xr = pick_nodes(x, y, xr)

            with torch.autocast(**autocast_kwargs):
                p = model(x)
                L, parts = split_loss(p, y, xr, grad_weight=grad_weight)

            total_losses.append(parts['total'])
            vel_losses.append(parts['vel'])
            grad_losses.append(parts['grad'])

            rels.append(float(rel_l2(p.float().unsqueeze(0), y.float().unsqueeze(0)).item()))
            mses.append(float(torch.mean((p.float() - y.float()) ** 2).item()))
            vel_rels.append(float(rel_l2(p[:, :3].float().unsqueeze(0), y[:, :3].float().unsqueeze(0)).item()))
            grad_rels.append(float(rel_l2(p[:, 3:].float().unsqueeze(0), y[:, 3:].float().unsqueeze(0)).item()))

    return {
        'rel_l2': float(np.mean(rels)),
        'mse': float(np.mean(mses)),
        'vel_rel_l2': float(np.mean(vel_rels)),
        'grad_rel_l2': float(np.mean(grad_rels)),
        'loss_total': float(np.mean(total_losses)),
        'loss_vel': float(np.mean(vel_losses)),
        'loss_grad': float(np.mean(grad_losses)),
    }


def eval_mean_baseline(loader, max_frames=64):
    # baseline in normalized space: predict mean train target
    ybar = torch.from_numpy(np.mean(Y_norm[train_rows], axis=0, keepdims=True)).float().to(DEVICE)
    rels = []
    with torch.no_grad():
        for b, (xs, ys, xrs, ms) in enumerate(loader):
            if b >= max_frames:
                break
            y = ys[0].to(DEVICE)
            pred = ybar.repeat(y.shape[0], 1)
            rels.append(float(rel_l2(pred.unsqueeze(0), y.unsqueeze(0)).item()))
    return float(np.mean(rels))


print('Smoke test...')
x0, y0, xr0, _ = train_ds[0]
x0, y0, xr0 = x0.to(DEVICE), y0.to(DEVICE), xr0.to(DEVICE)
x0, y0, xr0 = pick_nodes(x0, y0, xr0)
t0 = time.time()
with torch.autocast(**autocast_kwargs):
    p0 = model(x0)
    L0, parts0 = split_loss(p0, y0, xr0, grad_weight=GRAD_WEIGHT_STAGE1)
if scaler.is_enabled():
    scaler.scale(L0).backward()
else:
    L0.backward()
model.zero_grad(set_to_none=True)
print(f'Smoke OK | n={x0.shape[0]} | sec={time.time()-t0:.2f} | loss={L0.item():.6f}')

base_val = eval_mean_baseline(val_loader, max_frames=VAL_FRAMES_LIMIT)
print(f'Mean-baseline val rel-L2 (normalized target): {base_val:.6f}')

for ep in range(1, EPOCHS + 1):
    t_ep = time.time()
    model.train()

    # Stage-wise emphasis on gradient term
    grad_weight = GRAD_WEIGHT_STAGE1 if ep < GRAD_WEIGHT_STAGE2_EPOCH else GRAD_WEIGHT_STAGE2

    frame_total, frame_vel, frame_grad, frame_rel = [], [], [], []

    n_train = len(train_ds)
    use_n = min(FRAMES_PER_EPOCH, n_train)
    picks = np.random.choice(n_train, size=use_n, replace=False)

    for step, i in enumerate(picks, start=1):
        x, y, xr, _ = train_ds[int(i)]
        x, y, xr = x.to(DEVICE), y.to(DEVICE), xr.to(DEVICE)
        x, y, xr = pick_nodes(x, y, xr)

        opt.zero_grad(set_to_none=True)
        with torch.autocast(**autocast_kwargs):
            p = model(x)
            loss, parts = split_loss(p, y, xr, grad_weight=grad_weight)
            rel_step = rel_l2(p.float().unsqueeze(0), y.float().unsqueeze(0))

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            opt.step()

        frame_total.append(parts['total'])
        frame_vel.append(parts['vel'])
        frame_grad.append(parts['grad'])
        frame_rel.append(float(rel_step.item()))

        if step % PRINT_EVERY_STEPS == 0:
            print(
                f'[ep {ep:03d}] step {step:03d}/{use_n} '
                f'loss={parts["total"]:.6f} rel={rel_step.item():.6f} '
                f'vel={parts["vel"]:.6f} grad={parts["grad"]:.6f}',
                flush=True,
            )

    sch.step()

    train_loss = float(np.mean(frame_total))
    train_vel_loss = float(np.mean(frame_vel))
    train_grad_loss = float(np.mean(frame_grad))
    train_rel_l2 = float(np.mean(frame_rel))

    run_eval = (ep == 1) or (ep % EVAL_EVERY == 0) or (ep == EPOCHS)
    run_test = run_eval and ((ep == 1) or (ep % TEST_EVERY == 0) or (ep == EPOCHS))

    if run_eval:
        val = eval_loader(val_loader, max_frames=VAL_FRAMES_LIMIT, grad_weight=grad_weight)
    else:
        val = None

    if run_test:
        test = eval_loader(test_loader, max_frames=VAL_FRAMES_LIMIT, grad_weight=grad_weight)
    else:
        test = None

    row = {
        'epoch': ep,
        'lr': float(opt.param_groups[0]['lr']),
        'grad_weight': float(grad_weight),
        'train_loss': train_loss,
        'train_rel_l2': train_rel_l2,
        'train_vel_loss': train_vel_loss,
        'train_grad_loss': train_grad_loss,
        'val_rel_l2': float(val['rel_l2']) if val is not None else float('nan'),
        'val_mse': float(val['mse']) if val is not None else float('nan'),
        'val_vel_rel_l2': float(val['vel_rel_l2']) if val is not None else float('nan'),
        'val_grad_rel_l2': float(val['grad_rel_l2']) if val is not None else float('nan'),
        'val_loss_total': float(val['loss_total']) if val is not None else float('nan'),
        'val_loss_vel': float(val['loss_vel']) if val is not None else float('nan'),
        'val_loss_grad': float(val['loss_grad']) if val is not None else float('nan'),
        'test_rel_l2': float(test['rel_l2']) if test is not None else float('nan'),
        'test_mse': float(test['mse']) if test is not None else float('nan'),
        'test_vel_rel_l2': float(test['vel_rel_l2']) if test is not None else float('nan'),
        'test_grad_rel_l2': float(test['grad_rel_l2']) if test is not None else float('nan'),
        'epoch_sec': float(time.time() - t_ep),
    }
    history.append(row)

    if (val is not None) and (val['rel_l2'] < best_val):
        best_val = val['rel_l2']
        best_state = {k: v.detach().cpu() for k, v in model.state_dict().items()}

    if val is not None:
        test_msg = f" test={test['rel_l2']:.6f}" if test is not None else ' test=skipped'
        print(
            f"[epoch {ep:03d}] lr={row['lr']:.2e} train_loss={train_loss:.6f} train_rel={train_rel_l2:.6f} "
            f"val={val['rel_l2']:.6f} val_vel={val['vel_rel_l2']:.6f} "
            f"val_grad={val['grad_rel_l2']:.6f}{test_msg} sec={row['epoch_sec']:.1f}",
            flush=True,
        )
    else:
        print(
            f"[epoch {ep:03d}] lr={row['lr']:.2e} train_loss={train_loss:.6f} train_rel={train_rel_l2:.6f} "
            f"eval=skipped sec={row['epoch_sec']:.1f}",
            flush=True,
        )

    (OUT_DIR / 'history.json').write_text(json.dumps(history, indent=2))

if best_state is not None:
    model.load_state_dict(best_state)

# Always print final val/test on best checkpoint
final_val = eval_loader(val_loader, max_frames=VAL_FRAMES_LIMIT, grad_weight=GRAD_WEIGHT_STAGE2)
final_test = eval_loader(test_loader, max_frames=VAL_FRAMES_LIMIT, grad_weight=GRAD_WEIGHT_STAGE2)
print(f"Final(best) val_rel={final_val['rel_l2']:.6f} test_rel={final_test['rel_l2']:.6f}")

torch.save(model.state_dict(), OUT_DIR / 'best_task1_ugradu_gno.pt')
summary = {
    'notebook_version': NOTEBOOK_VERSION,
    'best_val_rel_l2': float(best_val),
    'final_val_rel_l2': float(final_val['rel_l2']),
    'final_test_rel_l2': float(final_test['rel_l2']),
    'mean_baseline_val_rel_l2': float(base_val),
    'epochs': EPOCHS,
    'device': str(DEVICE),
    'train_frames': len(train_ds),
    'val_frames': len(val_ds),
    'test_frames': len(test_ds),
    'geom_mode': GEOM_MODE,
    'loss_kind': LOSS_KIND,
}
(OUT_DIR / 'summary.json').write_text(json.dumps(summary, indent=2))
print('Saved summary:', json.dumps(summary, indent=2))



In [ ]:

# Evaluation snapshots + plots

def denorm_out(y_norm):
    return y_norm * out_std + out_mean

# Pick one test frame for plotting
x_norm, y_norm, x_raw_frame, meta = test_ds[0]
fr_idx = int(test_frame_ids[0])
s0, e0 = int(frame_ranges[fr_idx][2]), int(frame_ranges[fr_idx][3])
x_raw_full = X_all_raw[s0:e0]

if x_norm.shape[0] > MAX_NODES:
    if ('geom_dist' in feature_names):
        d = x_raw_frame[:, feature_names.index('geom_dist')].clamp_min(0.0)
        pr = torch.exp(-d / NEAR_SCALE)
        pr = pr / pr.sum().clamp_min(1e-12)
        idx = torch.multinomial(pr, num_samples=MAX_NODES, replacement=False)
    else:
        idx = torch.randperm(x_norm.shape[0])[:MAX_NODES]
    x_norm = x_norm[idx]
    y_norm = y_norm[idx]
    x_raw_frame = x_raw_frame[idx]

with torch.no_grad():
    y_pred_norm = model(x_norm.to(DEVICE)).cpu().numpy()

y_true = denorm_out(y_norm.numpy())
y_pred = denorm_out(y_pred_norm)
err = y_pred - y_true

# 1) loss curves
plt.figure(figsize=(9,4))
plt.plot([h['epoch'] for h in history], [h['train_loss'] for h in history], label='train_loss_total')
plt.plot([h['epoch'] for h in history], [h['val_loss_total'] for h in history], label='val_loss_total')
plt.plot([h['epoch'] for h in history], [h['val_loss_vel'] for h in history], label='val_loss_vel')
plt.plot([h['epoch'] for h in history], [h['val_loss_grad'] for h in history], label='val_loss_grad')
plt.grid(alpha=0.3); plt.legend(); plt.xlabel('epoch'); plt.ylabel('loss')
plt.title('Task1-u/gradU weighted losses')
plt.tight_layout(); plt.show()

# 2) relative error curves
plt.figure(figsize=(9,4))
plt.plot([h['epoch'] for h in history], [h['val_rel_l2'] for h in history], label='val_rel_l2')
plt.plot([h['epoch'] for h in history], [h['val_vel_rel_l2'] for h in history], label='val_vel_rel_l2')
plt.plot([h['epoch'] for h in history], [h['val_grad_rel_l2'] for h in history], label='val_grad_rel_l2')
plt.axhline(summary['mean_baseline_val_rel_l2'], color='k', ls='--', label='mean-baseline')
plt.grid(alpha=0.3); plt.legend(); plt.xlabel('epoch'); plt.ylabel('relative L2')
plt.title('Validation relative errors')
plt.tight_layout(); plt.show()

# 3) histogram of output errors
plt.figure(figsize=(7,4))
plt.hist(np.linalg.norm(err[:, :3], axis=1), bins=60, alpha=0.7, label='velocity error norm')
plt.hist(np.linalg.norm(err[:, 3:], axis=1), bins=60, alpha=0.7, label='gradU error norm')
plt.legend(); plt.xlabel('error norm'); plt.ylabel('count'); plt.title('Prediction error histograms')
plt.tight_layout(); plt.show()

# 4) per-feature relative error bar (one sampled test frame)
feat_rel = []
for j, name in enumerate(target_names):
    num = np.linalg.norm(err[:, j])
    den = np.linalg.norm(y_true[:, j]) + 1e-12
    feat_rel.append(num / den)

plt.figure(figsize=(10,4))
plt.bar(np.arange(len(target_names)), feat_rel)
plt.xticks(np.arange(len(target_names)), target_names, rotation=45, ha='right')
plt.ylabel('relative L2 (per feature)')
plt.title('Per-feature relative error (sampled frame)')
plt.tight_layout(); plt.show()

# 5) spatial plots on x-z plane
x = x_raw_frame.numpy()[:, 0]
z = x_raw_frame.numpy()[:, 2]

gamma_cols = [feature_names.index(k) for k in ['Gamma_x','Gamma_y','Gamma_z'] if k in feature_names]
sigma_col = feature_names.index('sigma') if 'sigma' in feature_names else None
gamma_mag = np.linalg.norm(x_raw_frame.numpy()[:, gamma_cols], axis=1) if len(gamma_cols)==3 else np.zeros_like(x)
sigma = x_raw_frame.numpy()[:, sigma_col] if sigma_col is not None else np.zeros_like(x)
vel_true = np.linalg.norm(y_true[:, :3], axis=1)
vel_pred = np.linalg.norm(y_pred[:, :3], axis=1)

fig, axs = plt.subplots(2, 3, figsize=(15, 9))
sc = axs[0,0].scatter(x, z, c=vel_true, s=2, cmap='turbo'); axs[0,0].set_title('True |u|'); fig.colorbar(sc, ax=axs[0,0])
sc = axs[1,0].scatter(x, z, c=vel_pred, s=2, cmap='turbo'); axs[1,0].set_title('Pred |u|'); fig.colorbar(sc, ax=axs[1,0])

sc = axs[0,1].scatter(x, z, c=gamma_mag, s=2, cmap='viridis'); axs[0,1].set_title('Input |Gamma|'); fig.colorbar(sc, ax=axs[0,1])
sc = axs[1,1].scatter(x, z, c=np.abs(vel_pred-vel_true), s=2, cmap='magma'); axs[1,1].set_title('|u| abs error'); fig.colorbar(sc, ax=axs[1,1])

sc = axs[0,2].scatter(x, z, c=sigma, s=2, cmap='plasma'); axs[0,2].set_title('Input sigma'); fig.colorbar(sc, ax=axs[0,2])
sc = axs[1,2].scatter(x, z, c=np.linalg.norm(err[:,3:], axis=1), s=2, cmap='magma'); axs[1,2].set_title('gradU error norm'); fig.colorbar(sc, ax=axs[1,2])

for a in axs.ravel():
    a.set_xlabel('x'); a.set_ylabel('z')
fig.tight_layout(); plt.show()

# Save arrays
np.savez_compressed(
    OUT_DIR / 'test_frame_predictions.npz',
    x_raw_frame=x_raw_frame.numpy(),
    y_true=y_true,
    y_pred=y_pred,
    error=err,
    feature_names=np.asarray(feature_names, dtype=object),
    target_names=np.asarray(target_names, dtype=object),
)
